# CIC IoT-DIAD 2024: in-depth exploratory data analysis

This notebook presents the compact artifacts generated by `iot-diad-audit` and `iot-diad-eda`. It does not read raw CSV files. Exact full-data results and sample-based descriptive results are identified explicitly throughout.

In [ ]:
import json
from pathlib import Path

import polars as pl
from IPython.display import Image, Markdown, display

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
audit_dir = project_root / "artifacts" / "data_audit"
eda_dir = project_root / "artifacts" / "eda"
figure_dir = eda_dir / "figures"

inventory = pl.read_csv(audit_dir / "file_inventory.csv")
class_summary = pl.read_csv(audit_dir / "class_summary.csv")
schema_summary = pl.read_csv(audit_dir / "schema_summary.csv")
column_profile = pl.read_csv(eda_dir / "column_profile.csv")
column_roles = pl.read_csv(eda_dir / "column_roles.csv")
duplicate_summary = pl.read_csv(eda_dir / "duplicate_summary.csv")
sample_duplicates = pl.read_csv(eda_dir / "sample_duplicate_summary.csv")
feature_summary = pl.read_csv(eda_dir / "feature_distribution_summary.csv")
correlations = pl.read_csv(eda_dir / "correlation_pairs.csv")
identifier_coverage = pl.read_csv(eda_dir / "identifier_coverage.csv")
packet_identifiers = pl.read_csv(eda_dir / "packet_identifier_profile.csv")
run_metadata = json.loads((eda_dir / "eda_run.json").read_text())
assert run_metadata["complete_run"], "The published notebook requires a complete EDA run."

## Evidence contract

Missingness, parse failures, non-finite values, numeric ranges, and within-capture duplicate counts cover every source row. Distribution summaries and correlations come from a capture-aware deterministic sample. Cross-capture duplicate results are risk diagnostics from that sample, not full-data leakage estimates.

In [ ]:
pl.DataFrame([run_metadata]).select(
    "selected_file_count",
    "complete_run",
    "seed",
    "sample_rows_per_capture",
    "analysis_rows_per_class",
)

## 1. Inventory, classes, and captures

In [ ]:
inventory.group_by("feature_view").agg(
    pl.len().alias("captures"),
    pl.col("row_count").sum().alias("rows"),
    pl.col("size_bytes").sum().alias("bytes"),
    pl.col("coarse_label").n_unique().alias("classes"),
    pl.col("row_count").median().alias("median_rows_per_capture"),
).sort("feature_view")

In [ ]:
display(Image(filename=figure_dir / "class_distribution.png", width=950))
display(Image(filename=figure_dir / "capture_size_distribution.png", width=950))

## 2. Schema consistency

In [ ]:
schema_summary.sort(["feature_view", "schema_status"])

## 3. Exact data-quality profile

The table below ranks columns by full-data missingness. Numeric parse failures and non-finite values are retained as separate failure modes.

In [ ]:
(
    column_profile.filter(pl.col("scope") == "overall")
    .sort(["feature_view", "missing_rate"], descending=[False, True])
    .group_by("feature_view", maintain_order=True)
    .head(15)
    .select(
        "feature_view", "column", "role", "missing_count",
        "missing_rate", "parse_failure_count", "nonfinite_count"
    )
)

In [ ]:
display(Image(filename=figure_dir / "missingness.png", width=950))

## 4. Feature roles and shortcut controls

Group identifiers, session identifiers, and source labels are excluded from model inputs. High-risk metadata is reserved for named ablations. Candidate categorical fields require train-only encoding and validation.

In [ ]:
column_roles.group_by(["feature_view", "role", "primary_baseline_action"]).agg(
    pl.len().alias("columns"),
    pl.col("column").sort().str.join(", ").alias("column_names"),
).sort(["feature_view", "role"])

## 5. Duplicate and leakage risk

Within-capture counts are exact. Candidate-feature duplicates are computed after removing explicit identifiers and high-risk metadata. Cross-capture and cross-class groups below are sample-based diagnostics; final train/test leakage can only be measured after grouped splitting.

In [ ]:
display(duplicate_summary.filter(pl.col("coarse_label") == "All"))
display(sample_duplicates)

In [ ]:
display(Image(filename=figure_dir / "duplicate_rates.png", width=700))

## 6. Feature distributions

Per-class summaries use at most 20,000 sampled rows per class to prevent DDoS and DoS from dominating descriptive comparisons. Quantiles are generally more informative than means for the heavy-tailed network measurements.

In [ ]:
representative = [
    "Flow Duration", "Flow Bytes/s", "Packet Length Mean",
    "inter_arrival_time", "eth_size", "payload_entropy",
]
feature_summary.filter(
    (pl.col("coarse_label") == "All") & pl.col("feature").is_in(representative)
).select(
    "feature_view", "feature", "sample_rows", "missing_rate",
    "p01", "q25", "median", "q75", "p99", "zero_rate"
).sort(["feature_view", "feature"])

In [ ]:
display(Image(filename=figure_dir / "representative_feature_quantiles.png", width=950))
display(Image(filename=figure_dir / "feature_distributions_flow.png", width=1050))
display(Image(filename=figure_dir / "feature_distributions_packet.png", width=1050))

## 7. Feature redundancy

Spearman correlations are computed from at most 50,000 balanced sampled rows per feature view. They describe monotone redundancy and do not imply causal relationships.

In [ ]:
correlations.group_by("feature_view", maintain_order=True).head(15).select(
    "feature_view", "left_feature", "right_feature",
    "spearman_correlation", "sample_rows"
)

In [ ]:
display(Image(filename=figure_dir / "correlation_flow.png", width=760))
display(Image(filename=figure_dir / "correlation_packet.png", width=760))

## 8. Device-identity feasibility

`device_mac` mixes MAC-form addresses, device names, and ambiguous text. Flow records expose private endpoints but no validated physical-device key. These fields support mapping work; they do not yet justify device-held-out claims.

In [ ]:
display(
    packet_identifiers.group_by("representation").agg(
        pl.len().alias("distinct_values"),
        pl.col("row_count").sum().alias("rows"),
        pl.col("capture_count").sum().alias("capture_occurrences"),
    ).sort("representation")
)
display(identifier_coverage)

In [ ]:
display(Image(filename=figure_dir / "packet_device_coverage.png", width=760))

## 9. Readiness for device-held-out evaluation

The schema, label provenance, and feature-exclusion policy are ready. Physical-device mapping remains the blocking dependency. The next artifact should be a manually validated table linking packet MAC/name representations and Flow IPs to canonical device IDs. Only then should the project construct S1/S2/S3 and train/calibration/test partitions.

In [ ]:
display(Markdown("Complete findings: [`eda_report.md`](../docs/reports/eda_report.md)"))